# Comment le projet se vérifie lui-même

Un mémoire d'ingénierie affirme beaucoup de choses. Ce notebook montre ce qui garantit qu'elles
restent vraies : les tests automatiques, ce qu'ils attrapent, et surtout ce qu'ils n'attrapent pas.

Il exécute la suite complète, montre le code de quelques tests, puis **en casse un volontairement**
pour voir ce qui se passe. Un garde-fou qu'on n'a jamais vu se déclencher n'est pas un garde-fou,
c'est une décoration.

## 1. Ce que le projet vérifie, et pourquoi

| Fichier | Ce qu'il garde | Ce qui arriverait sans lui |
|---|---|---|
| `test_socle.py` | version de Python, `.env.example` sans secret, un README par dossier, groupes de dépendances séparés | l'environnement dérive sans qu'on s'en aperçoive |
| `test_mesures_decp.py` | les requêtes SQL de mesure s'exécutent et renvoient les bons indicateurs | une faute de frappe dans un nom de colonne casserait la mesure en silence |
| `test_mesures_distributions.py` | les agrégats et les quinze figures se produisent sans erreur | une figure du mémoire planterait à la régénération |
| `test_chiffres_memoire.py` | chaque chiffre cité dans le mémoire correspond à une mesure enregistrée | le texte dirait autre chose que les données |

Un point de méthode : **aucun de ces tests ne lit le fichier de 236 Mo.** Ils travaillent soit sur
une table minuscule fabriquée pour l'occasion, soit sur les agrégats déjà versionnés. C'est ce qui
leur permet de tourner en quelques secondes, y compris sur la machine d'intégration continue, qui
n'a jamais téléchargé les données.

In [1]:
import subprocess
from pathlib import Path

RACINE = Path.cwd().parent


def lancer(*commande: str) -> str:
    """Exécute une commande dans le projet et renvoie sa sortie.

    `check=False` : on veut lire la sortie même quand la commande échoue, puisque c'est
    précisément ce que ce notebook cherche à montrer plus bas.
    """
    resultat = subprocess.run(  # noqa: S603
        commande, cwd=RACINE, capture_output=True, text=True, check=False
    )
    return resultat.stdout + resultat.stderr


print(lancer("uv", "run", "pytest", "--no-header", "-q", "--no-cov")[-1600:])

..................................................                       [100%]



## 2. Le test le plus important : les chiffres du mémoire

C'est celui qui rend exécutable la règle « aucun chiffre sans mesure ». Chaque valeur citée dans le
texte est reliée à la mesure qui la produit.

In [2]:
code = (RACINE / "tests" / "test_chiffres_memoire.py").read_text(encoding="utf-8")
debut = code.index("CHIFFRES_DU_MEMOIRE")
print(code[debut : code.index("]", code.index('"290752"')) + 1])

CHIFFRES_DU_MEMOIRE = [
    ("3 283 035 lignes brutes", lambda: qualite()["volume"]["lignes"], "3283035"),
    ("1 833 468 marches distincts", lambda: qualite()["lignes_actuelles"]["marches"], "1833468"),
    (
        "2 114 675 lignes a l'etat actuel",
        lambda: qualite()["lignes_actuelles"]["lignes"],
        "2114675",
    ),
    (
        "environ 7 800 doublons residuels",
        lambda: qualite()["origine_des_lignes_multiples"]["doublons_residuels"],
        "7764",
    ),
    (
        "273 494 groupements d'entreprises",
        lambda: qualite()["origine_des_lignes_multiples"]["multi_titulaires"],
        "273494",
    ),
    ("48 596 montants absents", lambda: qualite()["montants"]["absents"], "48596"),
    (
        "59 588 montants negatifs ou nuls",
        lambda: qualite()["montants"]["negatifs_ou_nuls"],
        "59588",
    ),
    (
        "2 197 montants au-dessus du milliard",
        lambda: qualite()["montants"]["superieurs_au_milliard"],
        "2197",
 

Le principe est simple : à gauche le chiffre tel qu'il est écrit dans le mémoire, au milieu la
mesure qui doit le confirmer, à droite la valeur attendue.

Si le jeu de données est retéléchargé demain et qu'une valeur bouge, le test échoue en nommant
précisément le chiffre devenu faux. C'est plus fiable que de relire quarante pages à la main.

## 3. Le voir échouer

On fabrique une copie du test avec **une valeur volontairement fausse**, on la lance, et on regarde
le message. C'est ce message qu'on lira le jour où une mesure changera réellement.

In [3]:
faux = code.replace('"3283035"', '"9999999"')
chemin_temporaire = RACINE / "tests" / "test_demonstration_echec.py"
chemin_temporaire.write_text(faux, encoding="utf-8")

sortie = lancer(
    "uv",
    "run",
    "pytest",
    "tests/test_demonstration_echec.py",
    # -k selectionne par mot-cle : un seul mot, sans espace, sinon pytest le lit
    # comme une expression logique et refuse la commande.
    "--no-header",
    "-q",
    "--no-cov",
    "-k",
    "brutes",
)
print(sortie[-1400:])

chemin_temporaire.unlink()  # on retire immédiatement le test volontairement faux

e ecrit dans le memoire doit correspondre a la mesure enregistree."""
        obtenu = mesure()
>       assert obtenu == attendu, (
            f"Le memoire annonce « {libelle} » mais la mesure donne {obtenu}. "
            "Mettre le texte a jour, ou verifier que la mesure est bien celle attendue."
        )
E       AssertionError: Le memoire annonce « 3 283 035 lignes brutes » mais la mesure donne 3283035. Mettre le texte a jour, ou verifier que la mesure est bien celle attendue.
E       assert '3283035' == '9999999'
E         
E         - 9999999
E         + 3283035

tests/test_demonstration_echec.py:97: AssertionError
=========================== short test summary info ============================
FAILED tests/test_demonstration_echec.py::test_chiffre_cite_dans_le_memoire[3 283 035 lignes brutes] - AssertionError: Le memoire annonce « 3 283 035 lignes brutes » mais la mesu...



Le message nomme le chiffre, donne la valeur mesurée, et rappelle quoi faire. C'est exactement le
comportement attendu : un test ne sert à rien s'il se contente de dire « échec ».

Le fichier de démonstration a été supprimé aussitôt, il n'entre pas dans le dépôt.

## 4. Les autres garde-fous du projet

Les tests ne sont qu'une partie. Quatre vérifications tournent à chaque commit et à chaque envoi.

In [4]:
for commande, role in (
    (("uv", "run", "ruff", "check", ".", "--statistics"), "style et pièges courants"),
    (("uv", "run", "mypy", ".", "--no-error-summary"), "cohérence des types"),
):
    print(f"--- {role} ---")
    print(lancer(*commande).strip() or "aucun problème")
    print()

--- style et pièges courants ---


aucun problème

--- cohérence des types ---


aucun problème



In [5]:
# gitleaks cherche des secrets dans les fichiers et dans l'historique des commits.
print(lancer("gitleaks", "dir", ".", "--no-banner", "--redact")[-400:])

5:48PM INF scanned ~249117779 bytes (249.12 MB) in 2m9s
5:48PM INF no leaks found



## 5. Ce que les tests n'attrapent pas

C'est la partie qu'un jury appréciera le plus, parce qu'elle montre qu'on connaît les limites de son
propre dispositif.

| Ce qui passe au travers | Pourquoi | Ce qui le rattrape |
|---|---|---|
| Une phrase juste sur les chiffres et fausse sur le fond | aucun test ne comprend le français | la relecture, et la soutenance |
| Une figure laide ou un axe mal choisi | le test vérifie que la figure se produit, pas qu'elle est lisible | regarder le fichier produit |
| Un indicateur statistiquement correct mais dénué de sens métier | rien dans le code ne connaît la commande publique | la confrontation aux sources et aux praticiens |
| Une source de données qui change de définition sans changer de format | les chiffres restent cohérents entre eux | la lecture des annonces du producteur |
| Un biais de sélection, comme les 56 % de marchés sans nombre d'offres | c'est une absence, pas une erreur | la mesure explicite du biais, à faire en phase 5 |

**Le cas le plus instructif est le dernier.** Les données manquantes ne déclenchent aucune alerte :
tout fonctionne, les calculs tournent, les figures se tracent. C'est précisément ce qui rend ce type
de défaut dangereux, et c'est pourquoi la complétude des champs a été mesurée et publiée avant
d'écrire la moindre ligne de détection.

## 6. Ce que ça coûte

| Vérification | Durée |
|---|---|
| Suite de tests complète | environ 15 secondes |
| Style et types | moins de 2 secondes |
| Recherche de secrets | moins d'une seconde |
| Chaîne d'intégration continue complète, sur GitHub | environ 30 secondes |

Pour ce prix, le projet garantit que ce qui est fusionné compile, passe ses tests, respecte son
style, ne contient aucun secret, et que les chiffres du mémoire correspondent toujours aux données.

C'est l'argument à opposer à l'idée reçue selon laquelle les tests coûtent du temps. Ce qui coûte
du temps, c'est de découvrir une erreur à la relecture finale, trois semaines après l'avoir écrite.